In [1]:
import requests
from pathlib import Path

vdv_consts_name = "const_vdv_rich.tsv"
vdv_consts_path = Path(f"/drive/data/{vdv_consts_name}")
vdv_consts_path.write_text(
    requests.get(
        f"https://cdn.lazarev.fun/data/{vdv_consts_name}"
    ).content.decode("utf8")
)


import pandas as pd

vdv_consts = pd.read_csv(vdv_consts_path, sep="\t")
vdv_consts.head(1)

,Substance,Formula,Const_a,Const_b,Mol_weight,Volume,cid,elements,atoms,bonds,...,cactvs_fingerprint,heavy_atom_count,isotope_atom_count,atom_stereo_count,defined_atom_stereo_count,undefined_atom_stereo_count,bond_stereo_count,defined_bond_stereo_count,undefined_bond_stereo_count,covalent_unit_count
0,Азот,N2,0.137,38.7,28.014,2.998428,947,"['N', 'N']","[{'aid': 1, 'number': 7, 'element': 'N', 'x': ...","[{'aid1': 1, 'aid2': 2, 'order': 3}]",...,0000000000000011000000000000000000000000000000...,2,0,0,0,0,0,0,0,1


In [5]:
vdv_consts_numbers = vdv_consts.select_dtypes(include="number")
vdv_consts_numbers.head(1)

,Const_a,Const_b,Mol_weight,Volume,cid,charge,molecular_weight,exact_mass,monoisotopic_mass,tpsa,...,rotatable_bond_count,heavy_atom_count,isotope_atom_count,atom_stereo_count,defined_atom_stereo_count,undefined_atom_stereo_count,bond_stereo_count,defined_bond_stereo_count,undefined_bond_stereo_count,covalent_unit_count
0,0.137,38.7,28.014,2.998428,947,0,28.014,28.006148,28.006148,47.6,...,0,2,0,0,0,0,0,0,0,1


In [6]:
const_a = vdv_consts_numbers["Const_a"]
const_b = vdv_consts_numbers["Const_b"]
vdv_consts_features = vdv_consts_numbers.drop(
    columns=["Const_a", "Const_b"]
)

In [8]:
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA

scaler = StandardScaler()
vdv_consts_features_transformed = \
    scaler.fit_transform(vdv_consts_features)

pca = PCA(n_components=len(vdv_consts_features.columns))
pca.fit(vdv_consts_features_transformed)
print(*pca.explained_variance_ratio_, sep="\n")

0.4436022946819001
0.19757820444489585
0.10497384137544895
0.09562393229113476
0.06896008043685227
0.052640444557808724
0.019733555558992272
0.008980320474511208
0.005047642869148966
0.002854013561350459
4.316676063543961e-06
1.3530718928596742e-06
2.5642558585327142e-33
1.1302668140549296e-33
0.0
0.0
0.0
0.0
0.0
0.0
0.0


In [10]:
from sklearn.linear_model import LinearRegression
reg = LinearRegression()
reg.fit(vdv_consts_features_transformed, const_b)
reg.score(vdv_consts_features_transformed, const_b)

0.7393855597541958

In [11]:
vdv_consts_features_transformed_pca = \
    pca.transform(vdv_consts_features_transformed)

In [12]:
reg = LinearRegression()
reg.fit(vdv_consts_features_transformed_pca, const_b)
reg.score(vdv_consts_features_transformed_pca, const_b)

0.7393855597541964

In [14]:
vdv_consts_features_transformed[:3, :3]

array([[-0.49275949, -0.4251223 , -0.73890706],
       [-0.77878878, -0.76223097, -0.7900094 ],
       [-0.18321345,  0.17141159,  0.88375108]])

In [15]:
vdv_consts_features_transformed_pca[:3, :3]

array([[-0.87399396,  1.91158791,  0.8230148 ],
       [-2.11129963,  0.20760605, -0.38612096],
       [-0.42528203, -1.23922877,  0.05791158]])

In [16]:
vdv_consts_features_transformed_pca.shape

(30, 21)

In [17]:
vdv_consts_features_transformed_pca_stricted = \
    vdv_consts_features_transformed_pca[:, :3]

In [18]:
vdv_consts_features_transformed_pca_stricted.shape

(30, 3)

In [19]:
reg = LinearRegression()
reg.fit(vdv_consts_features_transformed_pca_stricted, const_b)
reg.score(vdv_consts_features_transformed_pca_stricted, const_b)

0.4540868670141277

In [20]:
reg = LinearRegression()
reg.fit(vdv_consts_features_transformed_pca[:, :7], const_b)
reg.score(vdv_consts_features_transformed_pca[:, :7], const_b)

0.6091903837156667

In [22]:
reg = LinearRegression()
reg.fit(vdv_consts_features_transformed_pca[:, :7], const_a)
reg.score(vdv_consts_features_transformed_pca[:, :7], const_a)

0.6614559819793044

In [26]:
%pip install matplotlib
import matplotlib.pyplot as plt

_, ax = plt.subplots()
ax.scatter(
    vdv_consts_features_transformed_pca[:, 0],
    vdv_consts_features_transformed_pca[:, 2],
)


In [28]:
reg = LinearRegression()
reg.fit(vdv_consts_features_transformed_pca[:, :11], const_b)
reg.score(vdv_consts_features_transformed_pca[:, :11], const_b)

0.7344011495797993